In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 15:32:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Green

In [2]:
df_green = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("../../data/raw/green")
)  # read recursively 

In [3]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [4]:
df_green.registerTempTable("green")

/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [8]:
df_green_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    count(1) as number_records   
FROM
    green
where lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
-- order by 
--     1, 2
""")

# spark master showed 2 blockes of tasks for only group by , and 3 blocks of tasks for group by and order by 

In [9]:
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00| 244|183.57999999999998|            12|
|2020-01-01 00:00:00|  71|              23.8|             1|
|2020-01-01 02:00:00| 179|            206.82|            11|
|2020-01-01 02:00:00| 112|            417.35|            20|
|2020-01-01 04:00:00|  93|184.66000000000003|             2|
|2020-01-01 06:00:00| 193|              16.8|             1|
|2020-01-01 10:00:00|  25|             16.62|             1|
|2020-01-01 14:00:00| 196|110.88000000000001|             7|
|2020-01-02 06:00:00| 175|              37.0|             1|
|2020-01-02 10:00:00| 190|172.20000000000002|             3|
|2020-01-02 11:00:00|  98|               9.3|             1|
|2020-01-02 15:00:00| 146|            130.29|             3|
|2020-01-02 17:00:00| 171|              22.8|             1|
|2020-01-02 18:00:00|  5

In [10]:
df_green_revenue.write.parquet("../../data/report/revenue/green", mode="overwrite")

26/03/09 15:34:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/09 15:34:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/09 15:34:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/09 15:34:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/09 15:34:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/09 15:34:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/09 15:34:13 WARN MemoryManager: Total allocation exceeds 95.

# Yellow

In [13]:
df_yellow = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("../../data/raw/yellow")
)  # read recursively 

df_yellow.registerTempTable("yellow")

df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee']

In [17]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    count(1) as number_records   
FROM
    yellow
where tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
-- order by 
--     1, 2
""")

# spark master showed 2 blockes of tasks for only group by , and 3 blocks of tasks for group by and order by 

df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00| 244|183.57999999999998|            12|
|2020-01-01 00:00:00|  71|              23.8|             1|
|2020-01-01 02:00:00| 179|            206.82|            11|
|2020-01-01 02:00:00| 112|            417.35|            20|
|2020-01-01 04:00:00|  93|184.66000000000003|             2|
|2020-01-01 06:00:00| 193|              16.8|             1|
|2020-01-01 10:00:00|  25|             16.62|             1|
|2020-01-01 14:00:00| 196|110.88000000000001|             7|
|2020-01-02 06:00:00| 175|              37.0|             1|
|2020-01-02 10:00:00| 190|172.20000000000002|             3|
|2020-01-02 11:00:00|  98|               9.3|             1|
|2020-01-02 15:00:00| 146|            130.29|             3|
|2020-01-02 17:00:00| 171|              22.8|             1|
|2020-01-02 18:00:00|  5

In [18]:
df_yellow_revenue.write.parquet("../../data/report/revenue/yellow", mode="overwrite")
# alternatively (even though in practise it is ok if we have like 200 partitions if files are large 
# but here we could also simplify: 
# df_yellow_revenue.repartition(20).write.parquet("../../data/report/revenue/yellow", mode="overwrite")

26/03/09 15:39:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/09 15:39:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/09 15:39:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/09 15:39:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/09 15:39:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/09 15:39:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/09 15:39:33 WARN MemoryManager: Total allocation exceeds 95.